# Feature Engineering with PyTorch for CNN

This notebook demonstrates **Feature Engineering** using **Convolutional Neural Networks (CNNs)** for extracting features from image data.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision import models

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories for outputs
os.makedirs('../results/cnn_features', exist_ok=True)
os.makedirs('../models', exist_ok=True)

In [ ]:
# CNN Feature Extractor Architecture
class CNNFeatureExtractor(nn.Module):
    """
    CNN architecture specifically designed for feature extraction
    """
    def __init__(self, input_channels=3, feature_dim=512, num_classes=10):
        super(CNNFeatureExtractor, self).__init__()
        
        # Convolutional Feature Extraction Layers
        self.conv_layers = nn.Sequential(
            # Block 1
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.1),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.3),
            
            # Global Average Pooling
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # Feature extraction head
        self.feature_extractor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4)
        )
        
        # Classification head (optional)
        self.classifier = nn.Linear(feature_dim, num_classes)
        
        self.feature_dim = feature_dim
        
    def extract_features(self, x):
        """Extract features without classification"""
        conv_features = self.conv_layers(x)
        features = self.feature_extractor(conv_features)
        return features
        
    def forward(self, x):
        """Forward pass with classification"""
        features = self.extract_features(x)
        output = self.classifier(features)
        return output, features

# Transfer Learning Feature Extractor using Pre-trained ResNet
class ResNetFeatureExtractor(nn.Module):
    """
    Feature extractor using pre-trained ResNet with custom feature head
    """
    def __init__(self, feature_dim=512, num_classes=10, pretrained=True):
        super(ResNetFeatureExtractor, self).__init__()
        
        # Load pre-trained ResNet18
        self.backbone = models.resnet18(pretrained=pretrained)
        
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])
        
        # Custom feature extraction head
        self.feature_extractor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        
        # Classification head
        self.classifier = nn.Linear(feature_dim, num_classes)
        self.feature_dim = feature_dim
        
    def extract_features(self, x):
        """Extract features using pre-trained backbone"""
        backbone_features = self.backbone(x)
        features = self.feature_extractor(backbone_features)
        return features
        
    def forward(self, x):
        """Forward pass with classification"""
        features = self.extract_features(x)
        output = self.classifier(features)
        return output, features

# Test the models
print("CNN Feature Extractor:")
cnn_model = CNNFeatureExtractor(input_channels=3, feature_dim=256, num_classes=10)
print(f"Parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

print("\nResNet Feature Extractor:")
resnet_model = ResNetFeatureExtractor(feature_dim=256, num_classes=10)
print(f"Parameters: {sum(p.numel() for p in resnet_model.parameters()):,}")

# Test with sample input
sample_input = torch.randn(2, 3, 32, 32)
with torch.no_grad():
    cnn_output, cnn_features = cnn_model(sample_input)
    resnet_output, resnet_features = resnet_model(sample_input)
    
print(f"\nCNN Output shape: {cnn_output.shape}, Features shape: {cnn_features.shape}")
print(f"ResNet Output shape: {resnet_output.shape}, Features shape: {resnet_features.shape}")